# Nemotron 3 Ultra — Ollama Cloud

Snapshot-only experiment. No local Ollama, MSSQL refresh or provider fallback. Install `notebooks/nemotron/requirements.txt` first. Configure `OLLAMA_API_KEY` in your environment or ignored root `.env.nemotron`. Outputs must be cleared before saving/sharing.


## A. Configuration and Cloud model preflight

This cell checks availability with Cloud API; generation starts only in C. Missing credentials fail explicitly.


In [ ]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

workflow = None
verified_result = None
final_answer = None
REPO_ROOT = next(
    p
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "notebooks/shared/analytics.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
load_dotenv(REPO_ROOT / ".env.nemotron", override=False)
from notebooks.nemotron.adapter import NemotronClient, make_nemotron_workflow  # noqa: E402

preflight = NemotronClient()
try:
    print(json.dumps(preflight.check_model()))
finally:
    preflight.close()

## B. Open existing snapshot and discover approved schema

No refresh. Raw log text is blocked at the executor. Shared semantic checks are partial, not full ontology enforcement.


In [ ]:
if workflow is not None:
    workflow.client.close()
workflow = None
verified_result = None
final_answer = None
snapshot_path = Path(os.getenv("BENCHMARK_SQLITE_PATH", "self_healthy_kafka_snapshot.db"))
if not snapshot_path.is_absolute():
    snapshot_path = REPO_ROOT / snapshot_path
workflow = make_nemotron_workflow(snapshot_path)
print(
    json.dumps(
        {"snapshot": workflow.snapshot.metadata(), "schema": workflow.snapshot.schema},
        ensure_ascii=False,
        indent=2,
    )
)

## C. Question → verified SQL

Edit the question here. Maximum three SQL-stage calls including correction/review. Output is snapshot evidence, not live connector health.


In [ ]:
question = "Thống kê số lượng queue theo từng trạng thái QueueStatus hiện tại."
verified_result = None
final_answer = None
try:
    verified_result = workflow.query(question)
    print(
        json.dumps(
            {
                "verified_result": verified_result,
                "clarification": workflow.clarification,
                "trace": workflow.trace,
            },
            ensure_ascii=False,
            indent=2,
        )
    )
finally:
    print("SQL stage:", json.dumps(workflow.metrics, ensure_ascii=False))

## D. Vietnamese response from verified evidence

Rerun independently without re-executing SQL. Change the question only by rerunning C. A deterministic table fallback is explicitly labelled.


In [ ]:
final_answer = None
if workflow is None or workflow.question != question:
    raise RuntimeError("Question/configuration changed; run B and C before generating an answer.")
if workflow.result is None and not workflow.clarification:
    raise RuntimeError("No verified result; run C successfully first.")
final_answer = workflow.respond()
print("Response source:", final_answer["source"])
print(final_answer["text"])
print("Response and SQL metrics:", json.dumps(workflow.metrics, ensure_ascii=False))